# LLM ETL Pipeline

## Load Libraries

In [ ]:
import sys, os, json, re
from datetime import datetime

sys.path.insert(0, os.path.abspath(""))

from utils.config import *
from utils.llm_client import call_llm, LLM_PROVIDER
from utils.loader import load_tables
from utils.schema_description import build_schema_description, build_column_info
import pandas as pd

DW_DATA_DIR_LOCAL = "../data/4-data warehouse/"
LLM_PATH = ETL_REPORT_DIR / "LLM"
LLM_PATH.mkdir(parents=True, exist_ok=True)


def is_llm_error(text):
    return isinstance(text, str) and text.startswith("[LLM Error]")


print(f"LLM provider: {LLM_PROVIDER.upper()}")
print(f"Report output: {LLM_PATH}")

## Loading Artifacts

In [ ]:
dfs = load_tables("cleaned", normalize_cols="lower")

schema_path = MODELING_REPORT_DIR / "schema.json"
assert schema_path.exists(), f"Missing: {schema_path} -- run Notebook 7 first."

with open(schema_path) as f:
    schema = json.load(f)

star = schema["star_schema"]
fact_name = star["fact_table"]["name"]
measures = star["fact_table"]["measures"]
dim_names = list(star["dimension_tables"].keys())

print(f"Fact:      {fact_name}")
print(f"Measures:  {measures}")
print(f"Dimensions: {dim_names}")

In [ ]:
dw_tables = {}
for dim in dim_names:
    path = DW_DATA_DIR / f"{dim}.csv"
    if path.exists():
        dw_tables[dim] = pd.read_csv(path)
        print(f"[DW] {dim}: {len(dw_tables[dim]):,} rows x {len(dw_tables[dim].columns)} cols")
    else:
        print(f"[DW] {dim}: not found at {path}")

fact_path = DW_DATA_DIR / f"{fact_name}.csv"
if fact_path.exists():
    dw_tables[fact_name] = pd.read_csv(fact_path)
    print(f"[DW] {fact_name}: {len(dw_tables[fact_name]):,} rows x {len(dw_tables[fact_name].columns)} cols")

In [ ]:
# Build schema description for LLM prompts
schema_desc = build_schema_description(dfs, schema)
print(f"Schema description: {len(schema_desc):,} characters")

## ETL Plan Generator

In [ ]:
SYSTEM_ETL_ARCHITECT = """You are a senior ETL architect specializing in aviation Data Warehouses.
You design robust, idempotent ETL pipelines following dimensional modeling best practices.
Respond with clear, structured output. Focus on data quality and dependency management."""

prompt_etl_plan = f"""You are designing the complete ETL pipeline for a US domestic flight operations Data Warehouse.

STAR SCHEMA ARCHITECTURE:
- Fact Table: {fact_name}
- Measures: {measures}
- Dimensions: {dim_names}

Critical domain rules for loading sequence:
1. Generate MASTER TABLES FIRST: DIM_FL_DATES (365 calendar days), DIM_DEP_HOURS (0-23 hours)
2. Load DIMENSION HIERARCHIES: DIM_STATIONS (role-plays origin/dest), DIM_CARRIERS (role-plays mkt/op), DIM_AIRCRAFTS
3. Build DIM_JUNK DIMENSION: consolidate cancellation reason and active weather description
4. Load FACT TABLE LAST: ensure all foreign keys resolved, null handling in place

Edge cases to handle:
- Flights without usable weather observations are removed by the cleaning notebook before ETL; verify no unresolved Weather_Observation_ID remains
- Missing tail_num: resolve through DIM_AIRCRAFTS using Tail_Num; unresolved keys must be audited before fact loading
- Duplicate source records: apply deduplication strategy before loading
- Multiple role-playing dimensions: ensure DIM_STATIONS and DIM_CARRIERS are aliased correctly to avoid fan traps

For EACH step in the DAG, output EXACTLY:
[Step N] Load [TARGET_TABLE]
- Source(s): [source table names]
- Dependencies: [Steps this depends on]
- Key transforms: [dedup, surrogate key generation, SCD application]
- Data quality checks: [2-3 concrete checks]

Format as a numbered list. Be specific and actionable."""

etl_plan_response = call_llm(prompt_etl_plan, system=SYSTEM_ETL_ARCHITECT)
if is_llm_error(etl_plan_response):
    print(f"ETL plan generation failed -> {etl_plan_response}")
    etl_plan_response = ""
else:
    with open(LLM_PATH / "01-etl-plan.txt", "w", encoding="utf-8") as f:
        f.write(etl_plan_response)
    print(f"ETL plan saved to {LLM_PATH / '01-etl-plan.txt'}")
print(etl_plan_response)

In [ ]:
def parse_etl_dag(response: str) -> list[dict]:
    steps = []
    current = None
    for line in response.split("\n"):
        line = line.strip()
        if not line:
            continue
        normalized = line.replace("**", "").replace("`", "")
        step_match = re.search(r"Step\s*(\d+)\s*[:.)]?\s*Load\s*([A-Za-z0-9_]+)", normalized, re.IGNORECASE)
        if step_match:
            if current:
                steps.append(current)
            step_no = step_match.group(1)
            target = step_match.group(2).upper()
            current = {"step": f"Step {step_no}: Load {target}", "details": []}
            continue
        if current:
            current["details"].append(line)
    if current:
        steps.append(current)
    return steps


etl_dag = parse_etl_dag(etl_plan_response)
print(f"Parsed {len(etl_dag)} ETL steps from LLM response")
for i, step in enumerate(etl_dag, 1):
    print(f"  Step {i}: {step['step'][:80]}")

## Column Mapping Advisor (Station Table)

In [ ]:
station_df = dfs.get("Stations")
if station_df is not None:
    print(f"Source Station: {len(station_df):,} rows x {len(station_df.columns)} cols")
    station_col_info = build_column_info(station_df)
else:
    print("Station table not found.")
    station_col_info = ""

if "station_col_info" not in globals():
    station_col_info = build_column_info(station_df) if station_df is not None else ""

In [ ]:
dim_station_def = star["dimension_tables"].get("DIM_STATIONS", {})
print(f"Target DIM_STATIONS definition:")
print(json.dumps(dim_station_def, indent=2))

In [ ]:
if "station_col_info" not in globals():
    station_df = dfs.get("Stations")
    station_col_info = build_column_info(station_df) if station_df is not None else ""
if "dim_station_def" not in globals():
    dim_station_def = star["dimension_tables"].get("DIM_STATIONS", {})

SYSTEM_MAPPING_ANALYST = """You are a data mapping specialist for aviation warehouse projects.
Your job: map source columns to target dimension columns with precise SQL transformations.
Provide a clear, structured table showing each mapping, type conversion, and edge case handling.
Be specific enough that a junior analyst can implement this in SQL without ambiguity."""

prompt_column_mapping = f"""Task: Create an explicit column mapping from source Station table to target DIM_STATIONS dimension.

SOURCE TABLE: Station (reconciled data)
Available columns:
{station_col_info}

TARGET TABLE: DIM_STATIONS (Data Warehouse dimension)
Required structure:
{json.dumps(dim_station_def, indent=2)}

For EACH target column, provide:
1. Source column name (or state 'GENERATED' if computed)
2. SQL transformation (COALESCE, CASE, UPPER, etc.)
3. NULL handling strategy
4. Expected data type conversion
5. Business rules / domain notes

Domain context:
- AIRPORT: 3-letter FAA code = natural key; must be unique and non-null
- STATE: 2-letter ISO abbreviation (CA, NY, TX, etc.)
- CITY: Full city name; hierarchy level
- Hierarchy cardinality: Airport (many) -> City (fewer) -> State (50 max)
- Duplicate airports (e.g., different facilities in same city) -> keep all; distinguish by city

Format your response as a table with columns:
| Target_Column | Source_Column | Transformation | NULL_Strategy | Type | Notes |

Be extremely precise. Assume the reader has no context outside this table."""

column_mapping_response = call_llm(prompt_column_mapping, system=SYSTEM_MAPPING_ANALYST)
if is_llm_error(column_mapping_response):
    print(f"Column mapping generation failed -> {column_mapping_response}")
    column_mapping_response = ""
else:
    with open(LLM_PATH / "02-column-mapping-stations.txt", "w", encoding="utf-8") as f:
        f.write(column_mapping_response)
    print(f"Column mapping saved to {LLM_PATH / '02-column-mapping-stations.txt'}")
print(column_mapping_response)

In [ ]:
def parse_column_mapping(response: str) -> list[dict]:
    mappings = []
    for line in response.split("\n"):
        line = line.strip()
        if not line or line.startswith("#") or "---" in line:
            continue
        if "|" in line:
            parts = [p.strip() for p in line.split("|") if p.strip()]
            if len(parts) >= 2 and parts[0] not in ("Target", "Column"):
                mappings.append({"target": parts[0], "source": parts[1], "details": parts[2:]})
    return mappings


column_mappings = parse_column_mapping(column_mapping_response)
print(f"Parsed {len(column_mappings)} column mappings")
for m in column_mappings:
    print(f"  {m.get('target', '?'):25s} <- {m.get('source', '?')}")

## Pipeline Explainer

In [ ]:
etl_log_path = ETL_REPORT_DIR / "etl_pipeline_summary.csv"
if etl_log_path.exists():
    etl_log = pd.read_csv(etl_log_path)
    log_sample = etl_log.head(20).to_string(index=False)
    print(f"Using ETL log ({len(etl_log)} entries)")
else:
    log_entries = [
        {"step": "build_fl_dates", "status": "done", "output_rows": 365, "timestamp": "2024-01-15T02:00:01"},
        {"step": "build_dep_hours", "status": "done", "output_rows": 24, "timestamp": "2024-01-15T02:00:02"},
        {"step": "build_stations", "status": "done", "output_rows": 357, "timestamp": "2024-01-15T02:00:03"},
        {"step": "build_carriers", "status": "done", "output_rows": 14, "timestamp": "2024-01-15T02:00:04"},
        {"step": "build_junk", "status": "done", "output_rows": 30, "timestamp": "2024-01-15T02:00:05"},
        {"step": "build_fact_flight", "status": "done", "output_rows": 685502, "timestamp": "2024-01-15T02:00:08"},
    ]
    log_sample = pd.DataFrame(log_entries).to_string(index=False)
    print(f"Using synthetic ETL log ({len(log_entries)} entries)")

print(log_sample)

In [ ]:
SYSTEM_ETL_SUMMARIZER = """You are a data engineer communicating ETL results to business stakeholders.
Your task: translate technical audit logs into business-friendly summaries that explain
WHAT was loaded, HOW MANY rows, and WHETHER anything needs attention.
Use clear, professional tone. Avoid technical jargon. Be action-oriented."""

prompt_etl_summary = f"""Summarize this ETL execution log for business stakeholders. Focus on volume, timing, and any anomalies.

ETL EXECUTION LOG:
{log_sample}

Data Warehouse schema context:
- FLIGHTS (Fact): US domestic flights with departure delay, taxi time, air time, distance + weather context
- DIM_FL_DATES (Dimension): Calendar lookup; critical for time series analysis
- DIM_DEP_HOURS (Dimension): Hour-of-day (0-23); used to align weather snapshots
- DIM_STATIONS (Dimension): Airport dimension (role-plays as origin and destination)
- DIM_CARRIERS (Dimension): Airline codes (role-plays as marketing and operating carrier)
- DIM_AIRCRAFTS (Dimension): Aircraft registration, manufacturer, and age
- DIM_JUNK (Dimension): Cancellation reason and active weather condition

YOUR SUMMARY SHOULD INCLUDE:
1. Volume loaded (rows per table)
2. Data freshness (date range)
3. Any warnings or edge cases
4. Readiness for analytics
5. Recommended next step

Format: 3-4 short paragraphs. Be specific with numbers and dates. No jargon."""

etl_summary_response = call_llm(prompt_etl_summary, system=SYSTEM_ETL_SUMMARIZER)
if is_llm_error(etl_summary_response):
    print(f"ETL summary generation failed -> {etl_summary_response}")
    etl_summary_response = ""
else:
    with open(LLM_PATH / "03-etl-execution-summary.txt", "w", encoding="utf-8") as f:
        f.write(etl_summary_response)
    print(f"ETL summary saved to {LLM_PATH / '03-etl-execution-summary.txt'}")
print(etl_summary_response)

## SQL Generator

In [ ]:
SYSTEM_SQL_ENGINEER = """You are a senior SQL engineer writing production ETL code.
Your SQL must be: idempotent, performant, readable, and safe (no data loss, no duplicates).
Use PostgreSQL syntax. Include comments. Output ONLY the SQL code."""

prompt_sql_load_stations = f"""Write production-quality SQL to load the DIM_STATIONS dimension table.

Source: Station table (reconciled data)
Target: DIM_STATIONS dimension (star schema, no overwrites)

Source schema:
{station_col_info}

Target schema:
{json.dumps(dim_station_def, indent=2)}

Implementation requirements:
1. Use INSERT INTO ... SELECT (NOT UPSERT) to append new records
2. Generate Station_ID as SERIAL surrogate key
3. AIRPORT (3-letter code) is the natural key
4. Deduplicate on AIRPORT (if duplicates exist in source, keep first occurrence)
5. Filter: exclude rows where AIRPORT IS NULL
6. Enforce hierarchy: AIRPORT -> CITY -> STATE
7. Data quality: ensure STATE is 2-letter US abbreviation
8. Include clear SQL comments describing each step

Output:
- SQL only (no markdown, no explanations)
- Include a header comment with purpose, source, and target
- Add inline comments for complex logic
- Estimated row count and runtime notes (if relevant)

Optimize for readability and auditability."""

sql_load_stations = call_llm(prompt_sql_load_stations, system=SYSTEM_SQL_ENGINEER)
if is_llm_error(sql_load_stations):
    print(f"SQL generation failed -> {sql_load_stations}")
    sql_load_stations = ""
else:
    with open(LLM_PATH / "04-sql-load-stations.sql", "w", encoding="utf-8") as f:
        f.write(sql_load_stations)
    print(f"SQL saved to {LLM_PATH / '04-sql-load-stations.sql'}")
print(sql_load_stations)

In [ ]:
sql_checks = {
    "Has INSERT INTO": "INSERT INTO" in sql_load_stations.upper(),
    "Has SELECT": "SELECT" in sql_load_stations.upper(),
    "References Station": "station" in sql_load_stations.lower(),
    "Has DIM_STATIONS": "stations" in sql_load_stations.lower(),
    "Has Station_ID": "station_id" in sql_load_stations.lower(),
}

print("SQL validation:")
for check, passed in sql_checks.items():
    print(f"  {check}: {'PASS' if passed else 'FAIL'}")
all_passed = all(sql_checks.values())
print(f"Overall: {'All checks passed' if all_passed else 'Some checks failed'}")

## Pipeline Reviewer

In [ ]:
if "etl_plan_response" not in globals():
    etl_plan_path = LLM_PATH / "01-etl-plan.txt"
    etl_plan_response = etl_plan_path.read_text(encoding="utf-8") if etl_plan_path.exists() else ""
    print("Recovered etl_plan_response from disk." if etl_plan_response else "etl_plan_response not available; continuing with an empty ETL plan section.")

if "column_mapping_response" not in globals():
    mapping_path = LLM_PATH / "02-column-mapping-stations.txt"
    column_mapping_response = mapping_path.read_text(encoding="utf-8") if mapping_path.exists() else ""
    print("Recovered column_mapping_response from disk." if column_mapping_response else "column_mapping_response not available; continuing with an empty mapping section.")

pipeline_context = f"""STAR SCHEMA ARCHITECTURE:
  Fact Table: {fact_name}
  Measures: {measures}
  Dimensions: {dim_names}

DIMENSION DEFINITIONS:
"""
for dname, ddef in star["dimension_tables"].items():
    pipeline_context += f"\n{dname}:\n"
    pipeline_context += f"  Primary Key:  {ddef.get('surrogate_key', ddef.get('pk', 'N/A'))}\n"
    pipeline_context += f"  Hierarchy:    {ddef.get('levels', ddef.get('hierarchy', 'N/A'))}\n"
    pipeline_context += f"  Columns:      {ddef.get('columns', ddef.get('levels', []))}\n"

pipeline_context += f"""
ETL PLAN OUTLINE (from ETL Plan phase):
{etl_plan_response[:1500]}

COLUMN MAPPING DETAIL (DIM_STATIONS dimension):
{column_mapping_response[:1000]}
"""
print(f"Pipeline context prepared: {len(pipeline_context):,} characters")

In [ ]:
SYSTEM_PIPELINE_REVIEWER = """You are a senior data architect reviewing a warehouse ETL design.
Focus on correctness, edge cases, data quality, and operational risk.
For each issue: rate severity (HIGH/MEDIUM/LOW), explain impact, propose fix.
Be specific and actionable. Output: structured findings, not prose."""

prompt_pipeline_review = f"""Review this ETL pipeline design for a flight/weather Data Warehouse. Identify risks, edge cases, and design issues.

CONTEXT:
{pipeline_context}

CRITICAL AREAS TO REVIEW:

1. **NULL Handling in FLIGHTS (Fact table)**
   - Cleaning should remove flights without usable weather observations before ETL
   - Missing tail_num or aircraft not found -> audit before loading FLIGHTS
   - Risk: referential integrity violations or silent data loss

2. **Role-Playing Dimensions (FAN TRAP RISK)**
   - DIM_STATIONS appears twice: ORIGIN_AIRPORT_ID and DEST_AIRPORT_ID
   - DIM_CARRIERS appears twice: MKT_CARRIER_ID and OP_CARRIER_ID
   - Risk: if joined incorrectly, Cartesian explosion or wrong aggregation
   - Question: Are foreign keys properly aliased in queries?

3. **Time Dimension Consistency**
   - FL_DATE format (YYYY-MM-DD vs epoch vs datetime?)
   - Midnight-crossing flights: do they span two dates?
   - DIM_DEP_HOURS: derived as HOUR(scheduled_departure) or actual_departure?
   - Risk: time-series analysis errors, date mismatches

4. **Slowly Changing Dimension (SCD) Strategy**
   - Airport name changes (e.g., airport code renames) -> SCD Type 1 or 2?
   - Carrier code mergers (e.g., AA + US merger) -> how tracked?
   - Risk: historical analysis corrupted if SCD not applied

5. **DIM_JUNK Dimension Explosion**
   - Columns grouped: cancellation_reason (5+), weather_desc (10+)
   - Potential combinations: 5 x 10 = 50+ rows
   - Risk: sparse matrix; DIM_JUNK table becomes hard to maintain

6. **Duplicate Resolution**
   - If source Station has duplicate airport codes with different cities -> which wins?
   - If source Flight has duplicate flight_id on same_date -> which is canonical?
   - Risk: silent data loss; audit trail needed

For EACH finding, output:
[SEVERITY] Issue Name
- Description: (1-2 sentences)
- Business Impact: (what goes wrong if not fixed)
- Proposed Fix: (specific SQL/design change)

Rank by severity. Be direct and specific."""

pipeline_review_response = call_llm(prompt_pipeline_review, system=SYSTEM_PIPELINE_REVIEWER)
if is_llm_error(pipeline_review_response):
    print(f"Pipeline review generation failed -> {pipeline_review_response}")
    pipeline_review_response = ""
else:
    with open(LLM_PATH / "05-pipeline-review-findings.txt", "w", encoding="utf-8") as f:
        f.write(pipeline_review_response)
    print(f"Pipeline review saved to {LLM_PATH / '05-pipeline-review-findings.txt'}")
print(pipeline_review_response)

In [ ]:
def parse_review_findings(response: str) -> list[dict]:
    findings = []
    current = None
    for line in response.split("\n"):
        line = line.strip()
        if not line:
            continue
        sev_match = re.search(r"\b(HIGH|MEDIUM|LOW)\b", line.upper())
        if sev_match and (line.startswith("#") or line.startswith("*") or re.match(r"^\d+[.:)]", line) or any(kw in line for kw in ("Concern", "Issue", "Risk"))):
            if current:
                findings.append(current)
            current = {"title": line, "severity": sev_match.group(1), "details": []}
        elif current:
            current["details"].append(line)
    if current:
        findings.append(current)
    return findings


review_findings = parse_review_findings(pipeline_review_response)
severity_counts = {}
for f in review_findings:
    sev = f.get("severity", "UNKNOWN")
    severity_counts[sev] = severity_counts.get(sev, 0) + 1
    print(f"  [{sev}] {f['title'][:80]}")

print(f"\nSeverity summary: {severity_counts}")

## Save LLM Findings

In [ ]:
findings = {
    "notebook": "10-LLM ETL Pipeline",
    "generated_at": datetime.now().isoformat(),
    "llm_provider": LLM_PROVIDER,
    "star_schema": {
        "fact_table": fact_name,
        "measures": measures,
        "dimensions": dim_names,
    },
    "phase_1_etl_plan": {
        "description": "Complete ETL DAG with loading sequence and dependencies",
        "file_saved": "01-etl-plan.txt",
        "parsed_steps": len(etl_dag),
        "steps_summary": etl_dag,
    },
    "phase_2_column_mapping": {
        "description": "Explicit column-level mapping from source to DIM_STATIONS dimension",
        "target_table": "DIM_STATIONS",
        "file_saved": "02-column-mapping-stations.txt",
        "parsed_mappings": len(column_mappings),
        "mappings_summary": column_mappings,
    },
    "phase_3_etl_summary": {
        "description": "Business-friendly summary of ETL execution and data volume",
        "file_saved": "03-etl-execution-summary.txt",
    },
    "phase_4_sql_load_stations": {
        "description": "Production SQL to load DIM_STATIONS dimension table",
        "target_table": "DIM_STATIONS",
        "file_saved": "04-sql-load-stations.sql",
        "validation_checks": sql_checks,
        "all_checks_passed": all_passed,
    },
    "phase_5_pipeline_review": {
        "description": "Architecture review: edge cases, data quality risks, design issues",
        "file_saved": "05-pipeline-review-findings.txt",
        "parsed_findings": len(review_findings),
        "findings_summary": review_findings,
        "severity_distribution": severity_counts,
    },
}

output_path = LLM_PATH / "etl_pipeline_findings.json"
with open(output_path, "w") as f:
    json.dump(findings, f, indent=2, default=str)

print(f"Findings saved to: {output_path}")
print(f"File size: {os.path.getsize(output_path):,} bytes")

In [ ]:
print("\n" + "=" * 70)
print("LLM ETL PIPELINE -- EXECUTION SUMMARY")
print("=" * 70)
print(f"LLM Provider:         {LLM_PROVIDER.upper()}")
print(f"Star Schema:          {fact_name} + {len(dim_names)} dimensions")
print(f"\nPhase 1 (ETL Plan):         {len(etl_dag)} steps identified")
print(f"Phase 2 (Column Mapping):   {len(column_mappings)} mappings for DIM_STATIONS")
print(f"Phase 3 (ETL Summary):      Generated")
print(f"Phase 4 (SQL Generation):   {'All checks passed' if all_passed else 'Some checks failed'}")
print(f"Phase 5 (Architecture Review): {len(review_findings)} findings")
if severity_counts:
    sev_str = ", ".join([f"{k}={v}" for k, v in sorted(severity_counts.items(), reverse=True)])
    print(f"   Severity breakdown:  {sev_str}")
print(f"\nAll outputs saved to:      {LLM_PATH}")
print(f"Master findings file:       {output_path}")
print("=" * 70 + "\n")